# [GALAXY] – AstroScrappy CR Remover

<div class="alert alert-block alert-info">
<b>Environment:</b> Run this notebook in the <code>stenv</code> conda environment.
</div>

Remove cosmic rays from HST WFC3 FLC images using AstroScrappy (a Python implementation of the LA Cosmic algorithm).

## Imports

In [ ]:
# Python Imports
import os
from pathlib import Path

# Astropy Collaboration Imports
from astropy.io import fits

# Other Astronomy Imports
import astroscrappy

# 3rd Party Imports
from tqdm.notebook import tqdm


## Notebook Setup

In [ ]:
# Check Directory
if Path.cwd().name != "Images":
    if Path.cwd().name == "Notebooks":
        os.chdir("../Images")
    else:
        raise RuntimeError("This notebook must be run from the Images directory.")
print(f'Current Directory: {Path.cwd()}')

In [ ]:
# Data Directory
DATA_DIR = Path('RawImages/wfc3')

# FLC Glob Pattern
FLC_GLOB = DATA_DIR.rglob('*[!crclean]_fl?.fits')

## Store File Name Data

Read all FLC files and organize them into a dictionary keyed by filter.

In [ ]:
# Get the File Names and Sort them by filter
file_name_dict: dict = {}
for fn in FLC_GLOB:

    # Open the file to get the filter
    with fits.open(fn) as hdu_list:
        hdr = hdu_list[0].header  # Get the Header
        if 'FILTER' in hdr:      # If the FILTER keyword exists (WFC3)
            filt = hdr['FILTER']
        elif 'CLEAR' not in hdr['FILTER1']:  # If FILTER1 is not clear (ACS)
            filt = hdr['FILTER1']
        else:                                # Else FILTER2 must be the filter (ACS)
            filt = hdr['FILTER2']

    # Store the Name using the filter as the dict key
    # Start the Empty List if Key does not exist
    if filt not in file_name_dict:
        file_name_dict[filt] = []
    file_name_dict[filt].append(fn)

file_name_dict

## Remove CRs

Run AstroScrappy's LA Cosmic algorithm on each FLC image. Cleaned science arrays replace the original SCI extensions; the CR mask is stored in a new MSK extension and the DQ array is updated with flag 16384 (cosmic ray detected).

### Remove F814W CRs

In [ ]:
# Loop Through Files
for fn in tqdm(file_name_dict['F814W']):

    # Out Name
    out_name = fn.with_name(fn.name.replace('flc.fits', 'crclean_flc.fits'))

    # Open Image
    with fits.open(fn) as hdu_list:

        # Get Per-Chip Read Noise from Primary Header (averaged across amplifiers)
        # SCI,1 = UVIS2: amplifiers A & B; SCI,2 = UVIS1: amplifiers C & D
        phdr = hdu_list[0].header
        rdnoise1 = (phdr['READNSEA'] + phdr['READNSEB']) / 2
        rdnoise2 = (phdr['READNSEC'] + phdr['READNSED']) / 2

        # Get Cleaned Arrays
        crmsk1, crarr1 = astroscrappy.detect_cosmics(
            hdu_list['SCI', 1].data,
            # inmask=hdu_list['DQ', 1].data.astype(bool),
            invar=hdu_list['ERR', 1].data ** 2,
            gain=1.0,
            readnoise=rdnoise1,
            sigclip=4.5,
            sigfrac=0.3,
            objlim=5.0,
            satlevel=70000.0,
            niter=4,
            sepmed=True,
        )
        crmsk2, crarr2 = astroscrappy.detect_cosmics(
            hdu_list['SCI', 2].data,
            # inmask=hdu_list['DQ', 2].data.astype(bool),
            invar=hdu_list['ERR', 2].data ** 2,
            gain=1.0,
            readnoise=rdnoise2,
            sigclip=4.5,
            sigfrac=0.3,
            objlim=5.0,
            satlevel=70000.0,
            niter=4,
            sepmed=True,
        )

        # Write the Output Data
        out_list = hdu_list.copy()  # Copy the Original File
        out_list[0].header.add_history('CRs removed with AstroScrappy')
        out_list[0].header.add_comment('Created by Will Waldron, UAH')
        out_list[0].header.add_comment(
            'Created with pipeline at https://github.com/wwaldron/galred'
        )
        out_list['SCI', 1].data = crarr1.astype('float32')
        out_list['DQ', 1].data[crmsk1.astype(bool)] |= 16384
        out_list.insert(4, fits.ImageHDU(
            crmsk1.astype('uint8'), hdu_list['SCI', 1].header, 'MSK'
        ))
        out_list['SCI', 2].data = crarr2.astype('float32')
        out_list['DQ', 2].data[crmsk2.astype(bool)] |= 16384
        out_list.insert(8, fits.ImageHDU(
            crmsk2.astype('uint8'), hdu_list['SCI', 2].header, 'MSK'
        ))
        out_list.writeto(out_name, overwrite=True)


### Remove F475W CRs

In [ ]:
# Loop Through Files
for fn in tqdm(file_name_dict['F475W']):

    # Out Name
    out_name = fn.with_name(fn.name.replace('flc.fits', 'crclean_flc.fits'))

    # Open Image
    with fits.open(fn) as hdu_list:

        # Get Per-Chip Read Noise from Primary Header (averaged across amplifiers)
        # SCI,1 = UVIS2: amplifiers A & B; SCI,2 = UVIS1: amplifiers C & D
        phdr = hdu_list[0].header
        rdnoise1 = (phdr['READNSEA'] + phdr['READNSEB']) / 2
        rdnoise2 = (phdr['READNSEC'] + phdr['READNSED']) / 2

        # Get Cleaned Arrays
        crmsk1, crarr1 = astroscrappy.detect_cosmics(
            hdu_list['SCI', 1].data,
            # inmask=hdu_list['DQ', 1].data.astype(bool),
            invar=hdu_list['ERR', 1].data ** 2,
            gain=1.0,
            readnoise=rdnoise1,
            sigclip=4.5,
            sigfrac=0.3,
            objlim=5.0,
            satlevel=70000.0,
            niter=4,
            sepmed=True,
        )
        crmsk2, crarr2 = astroscrappy.detect_cosmics(
            hdu_list['SCI', 2].data,
            # inmask=hdu_list['DQ', 2].data.astype(bool),
            invar=hdu_list['ERR', 2].data ** 2,
            gain=1.0,
            readnoise=rdnoise2,
            sigclip=4.5,
            sigfrac=0.3,
            objlim=5.0,
            satlevel=70000.0,
            niter=4,
            sepmed=True,
        )

        # Write the Output Data
        out_list = hdu_list.copy()  # Copy the Original File
        out_list[0].header.add_history('CRs removed with AstroScrappy')
        out_list[0].header.add_comment('Created by Will Waldron, UAH')
        out_list[0].header.add_comment(
            'Created with pipeline at https://github.com/wwaldron/galred'
        )
        out_list['SCI', 1].data = crarr1.astype('float32')
        out_list['DQ', 1].data[crmsk1.astype(bool)] |= 16384
        out_list.insert(4, fits.ImageHDU(
            crmsk1.astype('uint8'), hdu_list['SCI', 1].header, 'MSK'
        ))
        out_list['SCI', 2].data = crarr2.astype('float32')
        out_list['DQ', 2].data[crmsk2.astype(bool)] |= 16384
        out_list.insert(8, fits.ImageHDU(
            crmsk2.astype('uint8'), hdu_list['SCI', 2].header, 'MSK'
        ))
        out_list.writeto(out_name, overwrite=True)
